# Chapter 7: Building Chat Applications
## OpenAI API Quickstart

This notebook is adapted from the [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) that includes notebooks that access [Azure OpenAI](notebook-azure-openai.ipynb) services.

The Python OpenAI API works with Azure OpenAI Models as well, with a few modifications. Learn more about the differences here: [How to switch between OpenAI and Azure OpenAI endpoints with Python](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)

# Overview  
"Large language models are functions that map text to text. Given an input string of text, a large language model tries to predict the text that will come next"(1). This "quickstart" notebook will introduce users to high-level LLM concepts, core package requirements for getting started with AML, a soft introduction to prompt design, and several short examples of different use cases. 

## Table of Contents  

[Overview](#overview)  
[How to use OpenAI Service](#how-to-use-openai-service)  
[1. Creating your OpenAI Service](#1.-creating-your-openai-service)  
[2. Installation](#2.-installation)    
[3. Credentials](#3.-credentials)  

[Use Cases](#use-cases)    
[1. Summarize Text](#1.-summarize-text)  
[2. Classify Text](#2.-classify-text)  
[3. Generate New Product Names](#3.-generate-new-product-names)  
[4. Fine Tune a Classifier](#4.fine-tune-a-classifier)  

[References](#references)

### Build your first prompt  
This short exercise will provide a basic introduction for submitting prompts to an OpenAI model for a simple task "summarization".


**Steps**:  
1. Install OpenAI library in your python environment  
2. Load standard helper libraries and set your typical OpenAI security credentials for the OpenAI Service that you've created  
3. Choose a model for your task  
4. Create a simple prompt for the model  
5. Submit your request to the model API!

### 1. Install OpenAI

In [2]:
%pip install openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


### 2. Import helper libraries and instantiate credentials

In [3]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Finding the right model  
The GPT-3.5-turbo or GPT-4 models can understand and generate natural language.

In [4]:
# Select the General Purpose curie model for text
model = "gpt-4o"

## 4. Prompt Design  

"The magic of large language models is that by being trained to minimize this prediction error over vast quantities of text, the models end up learning concepts useful for these predictions. For example, they learn concepts like"(1):

* how to spell
* how grammar works
* how to paraphrase
* how to answer questions
* how to hold a conversation
* how to write in many languages
* how to code
* etc.

#### How to control a large language model  
"Of all the inputs to a large language model, by far the most influential is the text prompt(1).

Large language models can be prompted to produce output in a few ways:

Instruction: Tell the model what you want
Completion: Induce the model to complete the beginning of what you want
Demonstration: Show the model what you want, with either:
A few examples in the prompt
Many hundreds or thousands of examples in a fine-tuning training dataset"



#### There are three basic guidelines to creating prompts:

**Show and tell**. Make it clear what you want either through instructions, examples, or a combination of the two. If you want the model to rank a list of items in alphabetical order or to classify a paragraph by sentiment, show it that's what you want.

**Provide quality data**. If you're trying to build a classifier or get the model to follow a pattern, make sure that there are enough examples. Be sure to proofread your examples — the model is usually smart enough to see through basic spelling mistakes and give you a response, but it also might assume this is intentional and it can affect the response.

**Check your settings.** The temperature and top_p settings control how deterministic the model is in generating a response. If you're asking it for a response where there's only one right answer, then you'd want to set these lower. If you're looking for more diverse responses, then you might want to set them higher. The number one mistake people make with these settings is assuming that they're "cleverness" or "creativity" controls.


Source: https://github.com/Azure/OpenAI/blob/main/How%20to/Completions.md

### 5. Submit!

In [5]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.chat.completions.create(
  model=model,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

'The use of the Oxford comma, also known as the serial comma, is a matter of style and preference rather than an absolute rule. It is used before the conjunction (usually \'and\' or \'or\') in a list of three or more items. For example:\n\n- With Oxford comma: "We packed apples, oranges, and bananas."\n- Without Oxford comma: "We packed apples, oranges and bananas."\n\nHere are some points to consider:\n\n**Arguments for using the Oxford comma:**\n1. **Clarity:** Sometimes the Oxford comma helps prevent ambiguity. For example: "I dedicate this book to my parents, Oprah Winfrey, and God." Without the Oxford comma, it could be interpreted as "I dedicate this book to my parents, Oprah Winfrey and God."\n2. **Consistency:** Some people prefer using the Oxford comma to maintain a consistent style, especially in longer, more complex lists.\n\n**Arguments against using the Oxford comma:**\n1. **Brevity:** Omitting the Oxford comma can make sentences slightly shorter and some believe it makes 

### Repeat the same call, how do the results compare?

In [6]:

response = client.chat.completions.create(
  model=model,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

'The use of the Oxford comma, also known as the serial comma, is a matter of style rather than a strict rule. It is the comma placed before the conjunction (usually "and" or "or") in a list of three or more items. For example: "I bought apples, oranges, and bananas."\n\n### Arguments For Using the Oxford Comma:\n1. **Clarity:** It can prevent misinterpretation. For example, "I thank my parents, Oprah Winfrey, and God" is clearer than "I thank my parents, Oprah Winfrey and God," where it might be read that Oprah Winfrey and God are the speaker\'s parents.\n2. **Consistency:** Using Oxford commas consistently can make your writing style uniform.\n\n### Arguments Against Using the Oxford Comma:\n1. **Brevity:** In some cases, especially in journalism and other writing styles that favor brevity, the Oxford comma is often omitted.\n2. **Tradition:** Certain style guides, such as the Associated Press (AP) style, traditionally do not use the Oxford comma.\n\n### Style Guides:\n- **Chicago Man

## Summarize Text  
#### Challenge  
Summarize text by adding a 'tl;dr:' to the end of a text passage. Notice how the model understands how to perform a number of tasks with no additional instructions. You can experiment with more descriptive prompts than tl;dr to modify the model’s behavior and customize the summarization you receive(3).  

Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches. 



Tl;dr

# Exercises for several use cases  
1. Summarize Text  
2. Classify Text  
3. Generate New Product Names

In [7]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [8]:
from rich import print
from rich.pretty import Pretty

#Setting a few additional, typical parameters during API Call

response = client.chat.completions.create(
  model=model,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

'Scaling up language models significantly enhances their few-shot performance on various NLP tasks, reducing the need for large task-specific fine-tuning datasets and sometimes matching the performance of state-of-the-art fine-tuned models.'

In [9]:
Pretty(response.choices[0].message.content)

'Scaling up language models significantly enhances their few-shot performance on various NLP tasks, reducing the 
need for large task-specific fine-tuning datasets and sometimes matching the performance of state-of-the-art 
fine-tuned models.'

## Classify Text  
#### Challenge  
Classify items into categories provided at inference time. In the following example, we provide both the categories and the text to classify in the prompt(*playground_reference). 

Customer Inquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:

Classified category:


In [10]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]

inquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:

Classified category:

In [11]:
#Setting a few additional, typical parameters during API Call

response = client.chat.completions.create(
  model=model,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

Pretty(response.choices[0].message.content)

'Classified category: Hardware Support'

## Generate New Product Names
#### Challenge
Create product names from examples words. Here we include in the prompt information about the product we are going to generate names for. We also provide a similar example to show the pattern we wish to receive. We have also set the temperature value high to increase randomness and more innovative responses.

Product description: A home milkshake maker
Seed words: fast, healthy, compact.
Product names: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Product description: A pair of shoes that can fit any foot size.
Seed words: adaptable, fit, omni-fit.

In [12]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

Product description: A home milkshake maker
Seed words: fast, healthy, compact.
Product names: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Product description: A pair of shoes that can fit any foot size.
Seed words: adaptable, fit, omni-fit.

In [15]:
import json

#Setting a few additional, typical parameters during API Call

response = client.chat.completions.create(
  model=model,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

# response.choices[0].message.content
# Pretty(response.choices[0].message.content)

# JSON print stuff!
json_data = json.loads(response.model_dump_json())
json_str = json.dumps(json_data, indent=3)
print(json_str)


{
   "id": "chatcmpl-9xYRW3cz0JJ6UjvNx1f4cHljoqzvJ",
   "choices": [
      {
         "finish_reason": "stop",
         "index": 0,
         "logprobs": null,
         "message": {
            "content": "1. **HomeShaker**\n   Create delicious and healthy milkshakes in no time with HomeShaker! 
This compact and powerful appliance is perfect for any kitchen, allowing you to blend your favorite ingredients 
effortlessly. Enjoy the convenience of making fast, nutritious shakes at home, any time you crave.\n\n2. **Fit 
Shaker**\n   Meet your new fitness companion - the Fit Shaker! Designed for health enthusiasts, this compact and 
speedy device helps you whip up nutrient-packed shakes without the fuss. Portable and efficient, the Fit Shaker 
ensures you get your daily dose of health, no matter how busy your schedule is.\n\n3. **QuickShake**\n   Experience
the joy of quick, healthy milkshakes with QuickShake! This compact powerhouse blends your shakes in seconds, making
it perfect for busy mornings or spontaneous treats. With QuickShake, you can enjoy delicious and nutritious blends 
without compromising on time or space.\n\n4. **Shake Maker**\n   Introducing the Shake Maker, your ultimate tool 
for creating fast and healthy milkshakes at home. Its compact design fits seamlessly into any kitchen, while its 
powerful functionality ensures your favorite shakes are ready in no time. Shake Maker \u2013 for the perfect blend 
of speed and wellness.\n\n---\n\n1. **AdaptFit Shoes**\n   Discover the ultimate in adaptable footwear with 
AdaptFit Shoes! These revolutionary shoes are designed to conform to any foot size, providing a perfect fit every 
time. Say goodbye to discomfort and hello to the seamless, omni-fit experience that moves with you.\n\n2. **OmniFit
Sneakers**\n   Step into the future with OmniFit Sneakers, the shoes that adapt to your feet! Perfectly designed to
fit any foot size, these adaptable sneakers ensure comfort and support throughout your day. Experience the next 
level of fit and functionality with OmniFit.\n\n3. **FlexiFit Shoes**\n   Introducing FlexiFit, the shoes that 
redefine adaptability. No matter your foot size, FlexiFit molds to provide the ideal fit, ensuring maximum comfort 
and support. Embrace the freedom of omni-fit flexibility and step out in style with FlexiFit Shoes.\n\n4. **FitAll 
Footwear**\n   FitAll Footwear brings you the unmatched convenience of shoes that adapt to any foot size, 
guaranteeing a perfect fit always. These innovative shoes embrace the concept of omni-fit, ensuring comfort and 
support for every step you take. FitAll \u2013 where adaptability meets elegance.",
            "role": "assistant",
            "function_call": null,
            "tool_calls": null,
            "refusal": null
         }
      }
   ],
   "created": 1723980778,
   "model": "gpt-4o-2024-05-13",
   "object": "chat.completion",
   "service_tier": null,
   "system_fingerprint": "fp_3aa7262c27",
   "usage": {
      "completion_tokens": 481,
      "prompt_tokens": 76,
      "total_tokens": 557
   }
}

# References  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [OpenAI Studio Examples](https://oai.azure.com/portal?WT.mc_id=academic-105485-koreyst)  
- [Best practices for fine-tuning GPT-3 to classify text](https://docs.google.com/document/d/1rqj7dkuvl7Byd5KQPUJRxc19BJt8wo0yHNwK84KfU3Q/edit#?WT.mc_id=academic-105485-koreyst)

# For More Help  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 

# Contributors
* Louis Li  
